# 2×2×2 Pocket Cube — Value Network Training (200K)

**Models:**
| Key | Architecture | Symmetry | Params |
|-----|-------------|----------|--------|
| `emlp_rot` | Group conv (spatial) | O, 24 spatial rotations | ~43K |
| `emlp_both` | Group conv (spatial+color) | O × S₆, 17280 elements | ~40K |
| `emlp_col` | MLP on color-matching matrix M[i,j]=x[i]·x[j] | S₆, 720 color perms (invariant) | ~38K |
| `mlp_aug` | MLP + data augmentation | None | ~40K |
| `mlp_matched` | MLP (parameter-matched) | None | ~40K |
| `mlp` | Large unconstrained MLP | None | ~169K |

**Why color-matching matrix for `emlp_col`?**
The S₆ group-conv approach (per-position processing + color-averaging) is provably constant-output:
for one-hot inputs, hidden activations depend only on which color is at each position, not on the
joint color assignment across positions. The color-matching matrix M is the *complete* S₆-invariant
descriptor — two states are color-equivalent iff M matches — so an MLP on M can learn anything.

**Train set:** 200K samples, seeds 42 / 43 / 44

### Setup (one-time)
1. Upload a Kaggle Dataset named **`cube-code`** containing these `.py` files:
   `cube_env.py`, `cube_group.py`, `cube_symmetry.py`, `equivariant_layers.py`,
   `models.py`, `dataset.py`, `train.py`, `evaluate.py`
2. Add that dataset to this notebook via **Add Data → Your Datasets**.
3. Enable GPU: **Settings → Accelerator → GPU T4 x2** (or P100).
4. Run All.

BFS and dataset files are generated inside the notebook (~10 min first run, cached after).

### Resuming a session
Upload the previous `results_export.zip` as a dataset named **`cube-results`** and add it here.
Cell 3 restores checkpoints, logs, and cached data so nothing is recomputed.

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
print('=== /kaggle/input contents ===')
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        print(f'{indent}  {f}')

In [ ]:
import os, shutil

WORK_DIR = '/kaggle/working'
DATA_DIR  = os.path.join(WORK_DIR, 'results', 'data')
CKPT_DIR  = os.path.join(WORK_DIR, 'results', 'checkpoints')
LOG_DIR   = os.path.join(WORK_DIR, 'results', 'logs')
PLOT_DIR  = os.path.join(WORK_DIR, 'results', 'plots')
for d in [DATA_DIR, CKPT_DIR, LOG_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

PY_FILES = {
    'cube_env.py', 'cube_group.py', 'cube_symmetry.py',
    'equivariant_layers.py', 'models.py',
    'dataset.py', 'train.py', 'evaluate.py',
}

copied = 0
for root, dirs, files in os.walk('/kaggle/input'):
    for fname in files:
        src = os.path.join(root, fname)
        if fname in PY_FILES:
            dst = os.path.join(WORK_DIR, fname)
        elif fname.endswith('.npy') or fname.endswith('.pkl'):
            dst = os.path.join(DATA_DIR, fname)
        elif fname.endswith('.pt'):
            dst = os.path.join(CKPT_DIR, fname)
        elif fname.endswith('.csv') and 'seed' in fname:
            dst = os.path.join(LOG_DIR, fname)
        else:
            continue
        shutil.copy2(src, dst)
        print(f'  copied → {os.path.relpath(dst, WORK_DIR)}')
        copied += 1

print(f'\nTotal: {copied} files copied.')

In [ ]:
import sys, os
WORK_DIR = '/kaggle/working'
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

# Only verify the Python source files here.
# Data files are generated (or restored) in the next cell.
py_files = [
    'cube_env.py', 'cube_group.py', 'cube_symmetry.py',
    'equivariant_layers.py', 'models.py',
    'dataset.py', 'train.py', 'evaluate.py',
]
all_ok = True
for f in py_files:
    ok = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'OK     ' if ok else 'MISSING'}  {f}")
    if not ok:
        all_ok = False

# Show any restored checkpoints
ckpt_dir = os.path.join(WORK_DIR, 'results', 'checkpoints')
ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
if ckpts:
    print(f'\nCheckpoints already present ({len(ckpts)}):')
    for f in ckpts:
        print(f'  {f}  ← will be skipped during training')

print()
print('Source files OK — proceed to data generation.' if all_ok else 'Fix missing .py files before continuing.')

In [ ]:
# ── BFS + dataset generation ──────────────────────────────────────────────────
# First run: BFS ~10 min, dataset generation ~5 min. Fully cached afterwards.
# If cube-results was restored in cell 3, everything is already present and skipped.
#
# Sampling strategy (set in train.py / dataset.py):
#   Training  — √-weighted: samples ∝ √(state count at depth).
#               Fixes equal-stratification's 52× oversampling of depth-14's 276 states.
#   Val/Test  — equal-stratified: 1000 per depth (unbiased early-stopping signal).
#
# √-weighted train files (X_train_*_sqrt.npy) are auto-generated by the first
# train.py call if missing — no manual step needed.
import os
DATA_DIR = os.path.join(WORK_DIR, 'results', 'data')

bfs_done = os.path.exists(os.path.join(DATA_DIR, 'bfs_tables.pkl'))
val_done = os.path.exists(os.path.join(DATA_DIR, 'X_val_strat.npy'))

if bfs_done and val_done:
    print("BFS table and base datasets already present — skipping generation.")
elif bfs_done:
    print("BFS cached. Generating datasets only (~5 min)...")
    !cd {WORK_DIR} && python dataset.py --skip-bfs
else:
    print("Running BFS + dataset generation (~15 min total)...")
    !cd {WORK_DIR} && python dataset.py

print("\nData files present:")
for f in sorted(os.listdir(DATA_DIR)):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"  {f:<45s}  {size_mb:6.1f} MB")

In [ ]:
# ── Train EMLP (spatial only, c=28 ~42K params) — 200K × 3 seeds ────────────
!cd {WORK_DIR} && python train.py --model emlp_rot --size 200k --seed 42
!cd {WORK_DIR} && python train.py --model emlp_rot --size 200k --seed 43
!cd {WORK_DIR} && python train.py --model emlp_rot --size 200k --seed 44

In [ ]:
# ── Train EMLP (spatial+color, k=20 ~40K params) — 200K × 3 seeds ──────────
!cd {WORK_DIR} && python train.py --model emlp_both --size 200k --seed 42
!cd {WORK_DIR} && python train.py --model emlp_both --size 200k --seed 43
!cd {WORK_DIR} && python train.py --model emlp_both --size 200k --seed 44

In [ ]:
# ── Train MLP (color-matching, h=60 ~38K params) — 200K × 3 seeds ───────────
# S6-invariant baseline: color-matching matrix M[i,j]=x[i,:]·x[j,:] fed into MLP.
# M is the COMPLETE S6-invariant descriptor — two states are color-equivalent iff M matches.
# Ablation: color-only invariance, no spatial equivariance.
!cd {WORK_DIR} && python train.py --model emlp_col --size 200k --seed 42
!cd {WORK_DIR} && python train.py --model emlp_col --size 200k --seed 43
!cd {WORK_DIR} && python train.py --model emlp_col --size 200k --seed 44

In [ ]:
# ── Train MLP baseline (h=256 ~169K params) — 200K × 3 seeds ───────────────
# Large unconstrained baseline: equivariant models win with ~4× fewer params.
!cd {WORK_DIR} && python train.py --model mlp --size 200k --seed 42
!cd {WORK_DIR} && python train.py --model mlp --size 200k --seed 43
!cd {WORK_DIR} && python train.py --model mlp --size 200k --seed 44

In [ ]:
# ── Train MLP+aug (h=110 ~40K params) — 200K × 3 seeds ─────────────────────
# Matched size to equivariant models: built-in vs learned symmetry, equal params.
# Applies random spatial rotation + color permutation to each batch.
!cd {WORK_DIR} && python train.py --model mlp_aug --size 200k --seed 42
!cd {WORK_DIR} && python train.py --model mlp_aug --size 200k --seed 43
!cd {WORK_DIR} && python train.py --model mlp_aug --size 200k --seed 44

In [ ]:
# ── Train MLP matched (h=110 ~40K params) — 200K × 3 seeds ─────────────────
# Parameter-matched to emlp (~42K): fair like-for-like baseline, no symmetry.
!cd {WORK_DIR} && python train.py --model mlp_matched --size 200k --seed 42
!cd {WORK_DIR} && python train.py --model mlp_matched --size 200k --seed 43
!cd {WORK_DIR} && python train.py --model mlp_matched --size 200k --seed 44

In [ ]:
# ── Evaluate all models ───────────────────────────────────────────────────────
!cd {WORK_DIR} && python evaluate.py --train-size 200000 \
    --models emlp_rot emlp_both emlp_col mlp_aug mlp_matched mlp

In [ ]:
# ── Display plots inline ──────────────────────────────────────────────────────
from IPython.display import Image, display
import os

PLOT_DIR = os.path.join(WORK_DIR, 'results', 'plots')
for fname in sorted(os.listdir(PLOT_DIR)):
    if fname.endswith('.png'):
        print(f'\n── {fname} ──')
        display(Image(os.path.join(PLOT_DIR, fname)))

In [ ]:
# ── Package results for download ──────────────────────────────────────────────
# Download BEFORE the session expires, then re-upload as a Kaggle dataset named
# "cube-results" to resume from checkpoints in the next session.
import shutil, os
shutil.make_archive('/kaggle/working/results_export', 'zip', WORK_DIR, 'results')
size_mb = os.path.getsize('/kaggle/working/results_export.zip') / 1e6
print(f'results_export.zip  ({size_mb:.1f} MB)')
print()
print('Kaggle output panel → results_export.zip → Download')
print()
print('Contents:')
for root, dirs, files in os.walk(os.path.join(WORK_DIR, 'results')):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, WORK_DIR)}  ({os.path.getsize(p)/1e3:.0f} KB)")